# sgf2gif + KataGo on Colab

This notebook clones the repo and runs a full end-to-end test step by step: unit tests, local rendering, remote downloads, KataGo analysis, and cache-based rerenders.

## 1. Install a compatible Go toolchain and clone the repository

Colab's `apt` package often ships an older Go release, so this notebook installs the official Go 1.26.1 toolchain directly.

In [ ]:
%cd /content
!apt-get -qq update
!apt-get -qq install -y curl tar
!rm -rf /usr/local/go
!curl -fsSL https://go.dev/dl/go1.26.1.linux-amd64.tar.gz -o /tmp/go1.26.1.linux-amd64.tar.gz
!tar -C /usr/local -xzf /tmp/go1.26.1.linux-amd64.tar.gz
import os
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']
os.chdir('/content')
!go version
!rm -rf /content/sgf2gif
!git clone https://github.com/rainoffallingstar/sgf2gif.git /content/sgf2gif

## 2. Configure Go caches (recommended)

This keeps module and build caches under `/content/` so reruns are faster and permissions are never an issue.

In [ ]:
import os
import pathlib

cache_root = pathlib.Path('/content/go-cache')
os.environ['GOMODCACHE'] = str(cache_root / 'gomodcache')
os.environ['GOCACHE'] = str(cache_root / 'gocache')
os.environ['GOPATH'] = str(cache_root / 'gopath')

for key in ['GOMODCACHE', 'GOCACHE', 'GOPATH']:
    pathlib.Path(os.environ[key]).mkdir(parents=True, exist_ok=True)

!go env GOPATH GOMODCACHE GOCACHE

## 3. Run unit tests

This verifies the repository is healthy before running any E2E steps.

In [ ]:
%cd /content/sgf2gif
!go test ./...

## 4. Build the `sgf2gif` binary

Build once, then reuse `./sgf2gif` for the remaining steps.

In [ ]:
%cd /content/sgf2gif
!go build -trimpath -ldflags="-s -w" -o ./sgf2gif .
!./sgf2gif -h | head -n 25

## 5. Local SGF -> GIF (no KataGo)

This confirms basic SGF parsing and rendering without any network or KataGo dependencies.

In [ ]:
%cd /content/sgf2gif
!./sgf2gif testdata/katago-e2e.sgf /content/local.gif
!test -s /content/local.gif
!ls -lh /content/local.gif

## 6. Pick a Fox qipu URL (optional)

If the default Fox page has expired, replace it with any current public `https://www.foxwq.com/qipu/newlist/id/...html` page.

In [ ]:
FOX_URL = 'https://www.foxwq.com/qipu/newlist/id/2026031862241631.html'
print('FOX_URL =', FOX_URL)

## 7. Choose KataGo backend (GPU vs CPU)

Use `auto` by default. If you hit `libcudnn.so.8` errors on T4, either install cuDNN and use `cuda`, or set `cpu` to run without GPU.

This step prints the current GPU and cuDNN detection to help you decide.

In [ ]:
KATAGO_BACKEND = 'auto'
print('KATAGO_BACKEND =', KATAGO_BACKEND)
!nvidia-smi || true
!ldconfig -p | grep -F libcudnn.so.8 || true

## 8. Remote SGF downloads + remote GIF render

This checks the downloader in three ways:

- download a single OGS game as SGF
- download a public Fox qipu page as SGF
- render a GIF directly from an OGS game URL without saving the SGF first

In [ ]:
%cd /content/sgf2gif
!./sgf2gif --download-sgf ogs:85130272 /content/ogs-download.sgf
!./sgf2gif --download-sgf {FOX_URL} /content/fox-download.sgf
!./sgf2gif https://online-go.com/game/85130272 /content/ogs-remote.gif
!test -s /content/ogs-download.sgf
!test -s /content/fox-download.sgf
!test -s /content/ogs-remote.gif
!ls -lh /content/ogs-download.sgf /content/fox-download.sgf /content/ogs-remote.gif
!sed -n '1,12p' /content/ogs-download.sgf
!sed -n '1,12p' /content/fox-download.sgf

## 9. KataGo analysis render (`mild`)

This runs a lightweight analysis and writes both the review GIF and a companion cache SGF at `*.katago.sgf`.

If you see a dynamic-loader error mentioning `libcudnn.so.8`, rerun step 7 and set `KATAGO_BACKEND = 'cpu'` (or install cuDNN and set it to `cuda`).

In [ ]:
%cd /content/sgf2gif
!./sgf2gif --katago-backend {KATAGO_BACKEND} --katago-strength mild testdata/katago-e2e.sgf /content/katago-mild.gif
!test -s /content/katago-mild.gif
!test -s /content/katago-mild.katago.sgf
!ls -lh /content/katago-mild.gif /content/katago-mild.katago.sgf

## 10. Cache rerender + cache-only check

This verifies that the companion `*.katago.sgf` can be used to rerender without rerunning KataGo, and that `--katago-cache-only` works.

In [ ]:
%cd /content/sgf2gif
!./sgf2gif /content/katago-mild.katago.sgf /content/katago-mild-rerender.gif
!./sgf2gif --katago-cache-only /content/katago-mild.katago.sgf /content/katago-cache-only.gif
!test -s /content/katago-mild-rerender.gif
!test -s /content/katago-cache-only.gif
!ls -lh /content/katago-mild-rerender.gif /content/katago-cache-only.gif

## 11. Preview GIFs

In [ ]:
from IPython.display import Image, display
import os

paths = [
    '/content/local.gif',
    '/content/ogs-remote.gif',
    '/content/katago-mild.gif',
    '/content/katago-mild-rerender.gif',
    '/content/katago-cache-only.gif',
    '/content/katago-fast.gif',
]

for path in paths:
    if os.path.exists(path):
        print(path)
        display(Image(filename=path))

## 12. Run a stronger preset (optional)

- `mild` = 50 visits
- `fast` = 100 visits
- `strong` = 1000 visits
- `monster` = 10000 visits

In [ ]:
%cd /content/sgf2gif
!./sgf2gif --katago-backend {KATAGO_BACKEND} --katago-strength fast testdata/katago-e2e.sgf /content/katago-fast.gif
!test -s /content/katago-fast.gif
!ls -lh /content/katago-fast.gif

## 13. Optional: custom visits / your own SGF

In [ ]:
%cd /content/sgf2gif
# Replace testdata/katago-e2e.sgf with your own uploaded SGF path if needed.
# !./sgf2gif --katago-analyze --katago-visits 500 --katago-threads 2 testdata/katago-e2e.sgf /content/katago-custom.gif